In [ ]:
# Prerequisite boilerplate
from pathlib import Path

# Tiny ImageNet as downloaded: ROOT/{train,test}/<class 000-199>/<image>.JPEG
ROOT = Path("../data/tiny-imagenet")
# Fixed seed, so the same train/val split is drawn on every run and machine
SEED = 5504132
TRAIN_FRACTION = 0.7


# The directory the split CSVs are written to, named after the train
#   fraction and the seed
def splits_file_name(train_split_fraction: float) -> Path:
    return (
        ROOT / "splits" / f"tiny-{train_split_fraction * 100:.0f}-"
        f"{(1 - train_split_fraction) * 100:.0f}_{SEED}"
    )

# Tiny ImageNet splits: 10 classes + other

An 11-way classification task: source classes `000`–`009` keep labels `0`–`9`, and the
other 190 become label `10` (`other`). This is meant to replicate the class imbalance
we expect to see in WSIs.

- **train / val:** the downloaded `train/` folder, split 70/30 within each source class.
- **test:** the downloaded `test/` folder (Tiny ImageNet's official validation set),
  kept for final reporting only (PLAN.md D14).

Writes CSVs to `OUT_FILE`: `{train,val,test}.csv` (`sample_id,path,label`, paths
relative to `ROOT`), `class_to_idx.csv` and `split.csv`. An existing split is never
overwritten (PLAN.md §3.5), because runs may already use it. To regenerate it, delete
the folder yourself first.

In [19]:
import pandas as pd

# Classes we want to keep
# These keep their own labels 0-9; the other 190 class folders become 'other'
KEEP = ["000", "001", "002", "003", "004", "005", "006", "007", "008", "009"]

# Name output file based on train split fraction
OUT_FILE = splits_file_name(TRAIN_FRACTION)

# Never overwrite an existing split: runs may already use it.
if OUT_FILE.exists():
    raise FileExistsError(f"{OUT_FILE} already exists; delete it to regenerate")

# Map classes to indexes (0-9 + 10 for 'other')
# `|` merges two dicts: {"000": 0, ..., "009": 9} | {"other": 10}
class_to_idx = {c: i for i, c in enumerate(KEEP)} | {"other": len(KEEP)}

## Split

In [20]:
def files(split: str) -> pd.DataFrame:
    # One row per image file, found at ROOT/<split>/<class>/<file>.
    # Sorted so the seeded sample below sees the same order on every platform.
    paths = sorted(p.relative_to(ROOT) for p in (ROOT / split).glob("*/*"))
    return pd.DataFrame(
        {
            # e.g. "train/003/train-..._12": the path without its extension
            "sample_id": [p.with_suffix("").as_posix() for p in paths],
            # relative to ROOT, with "/" separators on every OS
            "path": [p.as_posix() for p in paths],
            # the class folder the image came from, e.g. "003"
            "source": [p.parent.name for p in paths],
            # 0-9 for kept classes; .get falls back to 10 ('other') for the rest
            "label": [class_to_idx.get(p.parent.name, len(KEEP)) for p in paths],
        }
    )


pool = files("train")
# Draw 70% of the images of each of the 200 source classes, so 'other' is
# split evenly too, not just the 10 kept classes; random_state makes the
# draw repeatable
train = pool.groupby("source").sample(frac=TRAIN_FRACTION, random_state=SEED)
splits = {
    # back into file order (the sample shuffles rows)
    "train": train.sort_index(),
    # the remaining 30%: every pool row not drawn for train
    "val": pool.drop(train.index),
    # Tiny ImageNet's own validation set, used only for the final test
    "test": files("test"),
}

## Write

In [21]:
OUT_FILE.mkdir(parents=True)
# One manifest per split, with only the columns the Dataset reads
for name, df in splits.items():
    df[["sample_id", "path", "label"]].to_csv(OUT_FILE / f"{name}.csv", index=False)

# class_to_idx.csv: one row per class, with columns class,idx
pd.Series(class_to_idx, name="idx").rename_axis("class").to_csv(
    OUT_FILE / "class_to_idx.csv"
)
# split.csv: how the split was made, as key,value rows, so every manifest
# can be traced back to its seed and train fraction
pd.Series(
    {
        "root": ROOT.resolve().name,
        "seed": SEED,
        "train_fraction": TRAIN_FRACTION,
        "holdout": "per source class, from train/",
        "test": "test/ (official Tiny ImageNet validation set)",
    }
    | {f"{name}_size": len(df) for name, df in splits.items()},
    name="value",
).rename_axis("key").to_csv(OUT_FILE / "split.csv")

## Check

Read back what was written: one class map, disjoint splits, every source class split 70/30. The table is images per label.

In [22]:
# All three manifests in one table, with a column naming each row's split
written = pd.concat(
    [pd.read_csv(OUT_FILE / f"{n}.csv").assign(split=n) for n in splits],
    ignore_index=True,
)
# The class map read back matches the one used for the labels
#   (dtype=str stops "000" being read as the number 0)
classes = pd.read_csv(OUT_FILE / "class_to_idx.csv", dtype={"class": str})
assert dict(zip(classes["class"], classes["idx"], strict=True)) == class_to_idx

# No image is in two splits
assert written["sample_id"].is_unique, "a sample is in two splits"
# Every label is in 0-10
assert written["label"].between(0, len(KEEP)).all()
# Every path points at a real file
assert all((ROOT / p).is_file() for p in written["path"])

# Images per class folder ("train/003/..." -> "003") in each split, then
#   check train holds the train fraction of each class's train+val images
n = pd.crosstab(written["path"].str.split("/").str[1], written["split"])
assert (n["train"] == (TRAIN_FRACTION * (n["train"] + n["val"])).round()).all()

# Images per label in each split: the table shown below
pd.crosstab(written["label"], written["split"])

split,test,train,val
label,,,
0,50,350,150
1,50,350,150
2,50,350,150
3,50,350,150
4,50,350,150
5,50,350,150
6,50,350,150
7,50,350,150
8,50,350,150


## Class balance

`other` is 95% of every split (66,500 train images against 350 per kept class). The
manifests keep every image; balance is handled downstream:

- **train:** a `WeightedRandomSampler`, or class weights in the cross-entropy loss;
- **evaluate:** macro-averaged metrics (balanced accuracy, macro F1, per-class recall).
  Plain accuracy is 95% for a model that always predicts `other`.

## Load

A plain torch `Dataset` over a manifest, yielding the batch format of PLAN.md §3.1.

In [23]:
from torch.utils.data import DataLoader, Dataset
from torchvision.io import ImageReadMode, decode_image


class ManifestDataset(Dataset):
    def __init__(self, root: Path, manifest: Path):
        self.root = root
        # Each CSV row as a dict: {"sample_id": ..., "path": ..., "label": ...}
        self.rows = pd.read_csv(manifest).to_dict("records")

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, i: int) -> dict:
        row = self.rows[i]
        # uint8 [3, 64, 64]; RGB mode also converts Tiny ImageNet's few
        #   greyscale images to 3 channels, so every image stacks
        image = decode_image(str(self.root / row["path"]), mode=ImageReadMode.RGB)
        # One sample in the PLAN.md batch layout: inputs, targets and meta
        return {
            # floats in [0, 1]; normalisation belongs to the model
            "inputs": {"image": image.float() / 255},
            "targets": {"class": row["label"]},
            "meta": {"sample_id": row["sample_id"]},
        }


# DataLoader's default collate stacks 8 samples into one batch: images
#   become [8, 3, 64, 64], int labels a tensor of 8, string ids a list of 8
batch = next(
    iter(DataLoader(ManifestDataset(ROOT, OUT_FILE / "train.csv"), batch_size=8))
)
batch["inputs"]["image"].shape, batch["targets"]["class"]

(torch.Size([8, 3, 64, 64]), tensor([0, 0, 0, 0, 0, 0, 0, 0]))